%% [markdown]
# EndoSAM - Notebook 1: U-Net Experiments
All U-Net based experiments for pelvic organ segmentation.
Each cell below is one experiment. Run them in order.

**Experiments in this notebook:**
1. Data loading + alignment fix
2. Inter-rater agreement
3. Baseline U-Net (4-class) training
4. Attention U-Net (4-class) training
5. U-Net test evaluation
6. Binary ovary model training
7. Binary endometrioma model training
8. Binary test evaluation

## Cell 0: Imports and Config (run this first)

In [ ]:
import os, json, time, warnings, numpy as np, nibabel as nib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from PIL import Image
from collections import defaultdict
warnings.filterwarnings("ignore")

# ===== UPDATE THESE PATHS =====
DATASET_ROOT = "/path/to/UT-EndoMRI"
SAVE_DIR = os.path.expanduser("~/Desktop/EndoSAM_UNet_Compare")
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE = 256
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cpu")

# Find dataset folders
PREFIX_MAP = {}
for item in os.listdir(DATASET_ROOT):
    if os.path.isdir(os.path.join(DATASET_ROOT, item)):
        if item.startswith("D1"): PREFIX_MAP["D1"] = item
        elif item.startswith("D2"): PREFIX_MAP["D2"] = item
print(f"Dataset folders: {PREFIX_MAP}")

## Cell 1: Data Loading with Alignment Fix
The key discovery: 52% of subjects have masks annotated on a different
MRI sequence than the default image. We match each mask to the image
with the same spatial dimensions.

In [ ]:
def find_matching_image(subject_dir, subject_id, mask_shape):
    """Per-mask alignment fix: find image with same dimensions as mask"""
    for f in os.listdir(subject_dir):
        if not f.endswith(".nii.gz"): continue
        # Skip mask files
        if any(x in f for x in ["_ut", "_ov", "_em", "_cy", "_pat"]): continue
        img_path = os.path.join(subject_dir, f)
        img_shape = nib.load(img_path).shape
        # Match width and height
        if img_shape[0] == mask_shape[0] and img_shape[1] == mask_shape[1]:
            return img_path
    return None

def norm_pct(img_2d):
    """Percentile normalization: clip to 1st-99th percentile, scale to [0,1]"""
    fg = img_2d[img_2d > 0]
    if len(fg) == 0:
        return np.zeros_like(img_2d, dtype=np.float32)
    p1, p99 = np.percentile(fg, [1, 99])
    return np.clip((img_2d - p1) / (p99 - p1 + 1e-8), 0, 1).astype(np.float32)

# Collect all subjects
all_subjects = []
for pf in ["D1", "D2"]:
    fd = PREFIX_MAP.get(pf)
    if fd is None: continue
    pd = os.path.join(DATASET_ROOT, fd)
    for d in sorted(os.listdir(pd)):
        if d.startswith(pf + "-") and os.path.isdir(os.path.join(pd, d)):
            all_subjects.append(d)

# Subject-level split (same seed for all experiments)
idx = np.random.permutation(len(all_subjects))
n_tr = int(0.70 * len(all_subjects))
n_va = int(0.15 * len(all_subjects))
train_ids = set(all_subjects[i] for i in idx[:n_tr])
val_ids = set(all_subjects[i] for i in idx[n_tr:n_tr+n_va])
test_ids = set(all_subjects[i] for i in idx[n_tr+n_va:])
print(f"Total: {len(all_subjects)} subjects")
print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

def load_all_slices(subject_ids, struct_codes=["ut", "ov", "em", "cy"]):
    """Load all valid slices for given subjects with alignment fix"""
    data = []
    skipped = 0
    for sid in subject_ids:
        prefix = sid.split("-")[0]
        sdir = os.path.join(DATASET_ROOT, PREFIX_MAP[prefix], sid)
        if not os.path.isdir(sdir): continue
        
        for sc in struct_codes:
            # Find mask file (try multiple naming conventions)
            mask_path = None
            for c in [f"{sid}_{sc}.nii.gz", f"{sid}_{sc}_r3.nii.gz",
                      f"{sid}_{sc}_r2.nii.gz", f"{sid}_{sc}_r1.nii.gz",
                      f"{sid}_{sc}_re3.nii.gz"]:
                fp = os.path.join(sdir, c)
                if os.path.exists(fp):
                    mask_path = fp
                    break
            if mask_path is None: continue
            
            # Load mask
            mask_nib = nib.load(mask_path)
            mdata = mask_nib.get_fdata()
            if mdata.max() > 1 or np.unique(mdata).size > 2: continue
            mask = (mdata > 0.5).astype(np.uint8)
            
            # ALIGNMENT FIX: find image matching mask dimensions
            img_path = find_matching_image(sdir, sid, mask_nib.shape)
            if img_path is None:
                skipped += 1
                continue
            
            img_vol = nib.load(img_path).get_fdata().astype(np.float32)
            n_sl = min(img_vol.shape[2], mask.shape[2])
            
            for sl in range(n_sl):
                m2d = mask[:, :, sl]
                if m2d.sum() < 10: continue
                if img_vol[:, :, sl].max() == 0: continue
                data.append({
                    "sid": sid, "struct": sc, "slice": sl,
                    "img_vol": img_vol, "mask_vol": mask, "n_sl": n_sl
                })
    
    print(f"  Loaded {len(data)} slices, skipped {skipped} (no matching image)")
    return data

print("\nLoading training data...")
train_data = load_all_slices(train_ids)
print("Loading validation data...")
val_data = load_all_slices(val_ids)

## Cell 2: Inter-Rater Agreement
D1 has 3 raters. We compute pairwise Dice, excluding cross-sequence pairs.

In [ ]:
def compute_interrater():
    """Compute corrected inter-rater Dice for D1 subjects"""
    results = defaultdict(list)
    d1_dir = os.path.join(DATASET_ROOT, PREFIX_MAP["D1"])
    
    for sid in sorted(os.listdir(d1_dir)):
        if not sid.startswith("D1-"): continue
        sdir = os.path.join(d1_dir, sid)
        if not os.path.isdir(sdir): continue
        
        for struct in ["ut", "ov"]:
            # Find all rater files
            rater_masks = {}
            for f in os.listdir(sdir):
                if struct not in f: continue
                if not f.endswith(".nii.gz"): continue
                if "_r1" in f or "_r2" in f or "_r3" in f or "_re3" in f:
                    rater_id = f.split("_")[-1].replace(".nii.gz", "")
                else:
                    rater_id = "default"
                fp = os.path.join(sdir, f)
                nib_obj = nib.load(fp)
                rater_masks[rater_id] = {
                    "data": (nib_obj.get_fdata() > 0.5).astype(np.uint8),
                    "shape": nib_obj.shape
                }
            
            # Pairwise Dice (skip cross-sequence pairs with different dimensions)
            rater_list = list(rater_masks.keys())
            for i in range(len(rater_list)):
                for j in range(i+1, len(rater_list)):
                    r1 = rater_masks[rater_list[i]]
                    r2 = rater_masks[rater_list[j]]
                    # Skip if different dimensions (cross-sequence)
                    if r1["shape"] != r2["shape"]:
                        continue
                    m1 = r1["data"].flatten().astype(bool)
                    m2 = r2["data"].flatten().astype(bool)
                    if m1.sum() == 0 and m2.sum() == 0: continue
                    if m1.sum() == 0 or m2.sum() == 0:
                        d = 0.0
                    else:
                        d = 2 * (m1 & m2).sum() / (m1.sum() + m2.sum())
                    results[struct].append(float(d))
    
    for struct, dices in results.items():
        print(f"  {struct}: {np.mean(dices):.3f} +/- {np.std(dices):.3f} ({len(dices)} pairs)")

print("Inter-rater agreement (corrected):")
compute_interrater()

## Cell 3: PyTorch Dataset and Training Helpers

In [ ]:
STRUCT_TO_CH = {"ut": 1, "ov": 2, "em": 3, "cy": 3}

class EndoDataset(Dataset):
    """Multi-class dataset: 4 classes (bg=0, ut=1, ov=2, endo=3)"""
    def __init__(self, data_list):
        self.data = data_list
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        d = self.data[idx]
        sl = d["slice"]
        img_vol = d["img_vol"]
        mask_vol = d["mask_vol"]
        n_sl = d["n_sl"]
        h, w = img_vol.shape[:2]
        
        # 3-channel input: adjacent slices for context
        chs = []
        for off in [-1, 0, 1]:
            s = min(max(0, sl + off), n_sl - 1)
            chs.append(norm_pct(img_vol[:, :, s]))
        img_3 = np.stack(chs, axis=-1)
        
        # Resize
        img_r = np.array(Image.fromarray(
            (img_3 * 255).astype(np.uint8)
        ).resize((IMG_SIZE, IMG_SIZE))) / 255.0
        
        # Mask: assign class channel
        m2d = mask_vol[:, :, sl]
        ch = STRUCT_TO_CH[d["struct"]]
        label = np.zeros((h, w), dtype=np.int64)
        label[m2d > 0.5] = ch
        label_r = np.array(Image.fromarray(
            label.astype(np.uint8)
        ).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST))
        
        img_t = torch.from_numpy(img_r).permute(2, 0, 1).float()
        label_t = torch.from_numpy(label_r).long()
        return img_t, label_t

def dice_per_class(pred, gt, num_classes=4):
    """Compute Dice for each class"""
    dices = []
    for c in range(1, num_classes):  # skip background
        p = (pred == c).float()
        g = (gt == c).float()
        if g.sum() == 0 and p.sum() == 0:
            continue
        if g.sum() == 0 or p.sum() == 0:
            dices.append(0.0)
            continue
        d = (2 * (p * g).sum()) / (p.sum() + g.sum() + 1e-8)
        dices.append(d.item())
    return dices

## Cell 4: Baseline U-Net Training (4-class, cross-entropy)

In [ ]:
print("Training Baseline U-Net (4-class)...")
model = smp.Unet(encoder_name="efficientnet-b0", encoder_weights="imagenet",
                  in_channels=3, classes=4).to(device)

train_ds = EndoDataset(train_data)
val_ds = EndoDataset(val_data)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

best_dice = 0
for epoch in range(1, 16):
    model.train()
    t0 = time.time()
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        loss = criterion(out, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Validation
    model.eval()
    all_dices = []
    with torch.no_grad():
        for imgs, labels in val_loader:
            pred = model(imgs.to(device)).argmax(dim=1).cpu()
            all_dices.extend(dice_per_class(pred, labels))
    
    mean_dice = np.mean(all_dices) if all_dices else 0
    elapsed = time.time() - t0
    marker = ""
    if mean_dice > best_dice:
        best_dice = mean_dice
        torch.save({"model": model.state_dict(), "epoch": epoch},
                   os.path.join(SAVE_DIR, "vanilla_unet_best.pth"))
        marker = " <- BEST"
    print(f"  Epoch {epoch:2d} | Dice {mean_dice:.4f} | {elapsed:.0f}s{marker}")

print(f"Best validation Dice: {best_dice:.4f}")

## Cell 5: Attention U-Net Training (4-class with SCSE)

In [ ]:
print("Training Attention U-Net (4-class + SCSE)...")
model_attn = smp.Unet(encoder_name="efficientnet-b0", encoder_weights="imagenet",
                       in_channels=3, classes=4,
                       decoder_attention_type="scse").to(device)

optimizer = torch.optim.Adam(model_attn.parameters(), lr=1e-3)

best_dice = 0
for epoch in range(1, 16):
    model_attn.train()
    t0 = time.time()
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model_attn(imgs)
        loss = criterion(out, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    model_attn.eval()
    all_dices = []
    with torch.no_grad():
        for imgs, labels in val_loader:
            pred = model_attn(imgs.to(device)).argmax(dim=1).cpu()
            all_dices.extend(dice_per_class(pred, labels))
    
    mean_dice = np.mean(all_dices) if all_dices else 0
    elapsed = time.time() - t0
    marker = ""
    if mean_dice > best_dice:
        best_dice = mean_dice
        torch.save({"model": model_attn.state_dict(), "epoch": epoch},
                   os.path.join(SAVE_DIR, "attention_unet_best.pth"))
        marker = " <- BEST"
    print(f"  Epoch {epoch:2d} | Dice {mean_dice:.4f} | {elapsed:.0f}s{marker}")

print(f"Best validation Dice: {best_dice:.4f}")

## Cell 6: U-Net Test Evaluation (both models)

In [ ]:
print("Evaluating both U-Nets on test set...")
print("Loading test data...")
test_data_all = load_all_slices(test_ids)

for model_name in ["vanilla", "attention"]:
    model_path = os.path.join(SAVE_DIR, f"{model_name}_unet_best.pth")
    if not os.path.exists(model_path):
        print(f"  SKIP {model_name}: not found")
        continue
    
    if model_name == "vanilla":
        m = smp.Unet(encoder_name="efficientnet-b0", encoder_weights=None,
                     in_channels=3, classes=4).to(device)
    else:
        m = smp.Unet(encoder_name="efficientnet-b0", encoder_weights=None,
                     in_channels=3, classes=4,
                     decoder_attention_type="scse").to(device)
    
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    m.load_state_dict(ckpt["model"], strict=False)
    m.eval()
    
    results = []
    for d in test_data_all:
        sl = d["slice"]
        img_vol = d["img_vol"]
        mask_vol = d["mask_vol"]
        n_sl = d["n_sl"]
        h, w = img_vol.shape[:2]
        
        chs = [norm_pct(img_vol[:, :, min(max(0, sl+o), n_sl-1)]) for o in [-1, 0, 1]]
        img_3 = np.stack(chs, axis=-1)
        img_r = np.array(Image.fromarray((img_3*255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE)))/255.0
        img_t = torch.from_numpy(img_r).permute(2, 0, 1).float().unsqueeze(0)
        
        with torch.no_grad():
            pred = m(img_t).argmax(dim=1)[0].cpu().numpy()
        pred_orig = np.array(Image.fromarray(pred.astype(np.uint8)).resize((w, h), Image.NEAREST))
        
        ch = STRUCT_TO_CH[d["struct"]]
        p = (pred_orig == ch).astype(bool)
        g = (mask_vol[:, :, sl] > 0.5).astype(bool)
        if g.sum() == 0: continue
        dice = float(2 * (p & g).sum() / (p.sum() + g.sum() + 1e-8))
        results.append({"struct": d["struct"], "dice": round(dice, 4)})
    
    print(f"\n  {model_name} U-Net TEST ({len(results)} slices):")
    for s in ["ut", "ov", "em", "cy"]:
        e = [r["dice"] for r in results if r["struct"] == s]
        if e:
            print(f"    {s}: {np.mean(e):.3f} +/- {np.std(e):.3f} ({len(e)} slices)")

## Cell 7: Binary Ovary Model (2-class + Focal Tversky Loss)

In [ ]:
class BinaryDataset(Dataset):
    """Binary dataset for single organ: organ vs background"""
    def __init__(self, data_list, target_struct):
        self.data = [d for d in data_list if d["struct"] == target_struct]
        print(f"    Binary {target_struct}: {len(self.data)} slices")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        d = self.data[idx]
        sl, img_vol, mask_vol, n_sl = d["slice"], d["img_vol"], d["mask_vol"], d["n_sl"]
        chs = [norm_pct(img_vol[:, :, min(max(0, sl+o), n_sl-1)]) for o in [-1, 0, 1]]
        img_3 = np.stack(chs, axis=-1)
        img_r = np.array(Image.fromarray((img_3*255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE)))/255.0
        m2d = (mask_vol[:, :, sl] > 0.5).astype(np.uint8)
        m_r = np.array(Image.fromarray(m2d).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST))
        return torch.from_numpy(img_r).permute(2, 0, 1).float(), torch.from_numpy(m_r).long()

class FocalTverskyLoss(nn.Module):
    """alpha=0.8 penalizes FN 4x more than FP. gamma=1.33 focuses on hard examples."""
    def __init__(self, alpha=0.8, beta=0.2, gamma=1.33):
        super().__init__()
        self.alpha, self.beta, self.gamma = alpha, beta, gamma
        self.ce = nn.CrossEntropyLoss()
    
    def forward(self, pred, target):
        prob = torch.softmax(pred, dim=1)[:, 1]  # organ channel
        gt = (target == 1).float()
        tp = (prob * gt).sum()
        fn = ((1 - prob) * gt).sum()
        fp = (prob * (1 - gt)).sum()
        ti = (tp + 1e-8) / (tp + self.alpha * fn + self.beta * fp + 1e-8)
        ftl = (1 - ti) ** self.gamma
        return 0.7 * ftl + 0.3 * self.ce(pred, target)

def train_binary(target_struct, save_name, epochs=15):
    print(f"\nTraining binary {target_struct}...")
    model = smp.Unet(encoder_name="efficientnet-b0", encoder_weights="imagenet",
                     in_channels=3, classes=2).to(device)
    
    tr_ds = BinaryDataset(train_data, target_struct)
    va_ds = BinaryDataset(val_data, target_struct)
    tr_loader = DataLoader(tr_ds, batch_size=8, shuffle=True, num_workers=0)
    va_loader = DataLoader(va_ds, batch_size=8, shuffle=False, num_workers=0)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    criterion = FocalTverskyLoss()
    
    best_dice = 0
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        for imgs, labels in tr_loader:
            out = model(imgs.to(device))
            loss = criterion(out, labels.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        model.eval()
        dices = []
        with torch.no_grad():
            for imgs, labels in va_loader:
                pred = model(imgs.to(device)).argmax(dim=1).cpu()
                p = (pred == 1).float().flatten()
                g = (labels == 1).float().flatten()
                if g.sum() > 0:
                    d = (2 * (p * g).sum() / (p.sum() + g.sum() + 1e-8)).item()
                    dices.append(d)
        
        md = np.mean(dices) if dices else 0
        marker = ""
        if md > best_dice:
            best_dice = md
            torch.save({"model": model.state_dict()}, save_name)
            marker = " <- BEST"
        print(f"  Epoch {epoch:2d} | Dice {md:.4f} | {time.time()-t0:.0f}s{marker}")
    
    print(f"Best: {best_dice:.4f}")

SAVE_OV = os.path.expanduser("~/Desktop/EndoSAM_BinaryOvary/best_binary_ovary.pth")
SAVE_EN = os.path.expanduser("~/Desktop/EndoSAM_BinaryEndo/best_binary_endo.pth")
os.makedirs(os.path.dirname(SAVE_OV), exist_ok=True)
os.makedirs(os.path.dirname(SAVE_EN), exist_ok=True)

train_binary("ov", SAVE_OV)

## Cell 8: Binary Endometrioma Model

In [ ]:
train_binary("em", SAVE_EN)

## Cell 9: Binary Test Evaluation

In [ ]:
def test_binary(model_path, target_struct):
    model = smp.Unet(encoder_name="efficientnet-b0", encoder_weights=None,
                     in_channels=3, classes=2).to(device)
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])
    model.eval()
    
    test_struct = [d for d in test_data_all if d["struct"] == target_struct]
    results = []
    for d in test_struct:
        sl = d["slice"]
        img_vol, mask_vol, n_sl = d["img_vol"], d["mask_vol"], d["n_sl"]
        h, w = img_vol.shape[:2]
        chs = [norm_pct(img_vol[:, :, min(max(0, sl+o), n_sl-1)]) for o in [-1, 0, 1]]
        img_3 = np.stack(chs, axis=-1)
        img_r = np.array(Image.fromarray((img_3*255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE)))/255.0
        img_t = torch.from_numpy(img_r).permute(2, 0, 1).float().unsqueeze(0)
        with torch.no_grad():
            pred = model(img_t).argmax(dim=1)[0].cpu().numpy()
        pred_orig = np.array(Image.fromarray(pred.astype(np.uint8)).resize((w, h), Image.NEAREST))
        p = (pred_orig == 1).astype(bool)
        g = (mask_vol[:, :, sl] > 0.5).astype(bool)
        if g.sum() == 0: continue
        dice = float(2 * (p & g).sum() / (p.sum() + g.sum() + 1e-8))
        results.append({"dice": round(dice, 4), "predicted": bool(p.sum() > 0)})
    
    all_d = [r["dice"] for r in results]
    pred_d = [r["dice"] for r in results if r["predicted"]]
    print(f"  {target_struct}: {len(results)} slices")
    print(f"    Overall mean: {np.mean(all_d):.3f}")
    if pred_d:
        print(f"    When predicted: {np.mean(pred_d):.3f} ({len(pred_d)}/{len(results)} detected)")
    return results

print("Binary TEST evaluation:")
test_binary(SAVE_OV, "ov")
test_binary(SAVE_EN, "em")